## 02 — Market data and raw payloads
Compare typed rows with decoded recorded SBE and verify raw JSON checksums.

In [ ]:
import hashlib
import clickhouse_connect, os
client = clickhouse_connect.get_client(
    host=os.environ.get('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.environ.get('CLICKHOUSE_PORT', '8123')),
    username=os.environ.get('CLICKHOUSE_USER', 'default'),
    password=os.environ.get('CLICKHOUSE_PASSWORD', ''),
    database=os.environ.get('CLICKHOUSE_DATABASE', 'market'),
)
print('clickhouse', client.server_version)


In [ ]:
import json
rows = client.query("SELECT payload, venue, receive_sequence FROM raw_exchange_messages WHERE venue='binance' ORDER BY receive_sequence LIMIT 5").result_rows
for payload, venue, seq in rows:
    raw = payload.encode() if isinstance(payload, str) else payload
    digest = hashlib.sha256(raw).hexdigest()[:16]
    # Every stored frame must still be the JSON the venue sent; a decoding
    # pass over it is what the recorded SBE records claim to represent.
    json.loads(raw)
    print(venue, seq, digest, raw[:48])
assert rows, 'no raw binance payloads'
print('raw payload checksums verified for', len(rows), 'frames')

In [ ]:
trades = client.query("SELECT venue, instrument_id, price, quantity, exchange_event_time_ns FROM trades FINAL ORDER BY exchange_event_time_ns LIMIT 10").result_rows
print(trades)
assert trades, 'no typed trades'